# Ordered Logistic Regression Results on Rangeland Management – Exploration with `mlcroissant`
This notebook provides a reproducible template for loading and exploring the FAIR² dataset (Northern Kenya Ordered Logistic Regression Results) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset's Croissant metadata is available at:
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset Croissant object
dataset = mlc.Dataset(croissant_url)

# Access and print dataset metadata
metadata = dataset.metadata
print(f"Name: {metadata.name}")
print(f"Identifier: {metadata.identifier}")
print(f"Description: {metadata.description}")
print(f"Authors: {getattr(metadata, 'author', None)}")
print(f"Published: {getattr(metadata, 'datePublished', None)}")

## 2. Data Overview
Review available record sets, fields, and their unique Croissant `@id`s.

In [ ]:
# List record sets
print("Available record sets (by @id):")
record_sets = dataset.record_sets()
record_set_ids = []
for rs in record_sets:
    print(f"- {rs['@id']}  (name: {rs.get('name', None)})")
    record_set_ids.append(rs['@id'])

# Get example fields/columns in the first available record set
if record_set_ids:
    example_rs_id = record_set_ids[0]
    print(f"\nFields for record set '{example_rs_id}':")
    fields = dataset.fields(record_set=example_rs_id)
    for field in fields:
        print(f"  - field @id: {field['@id']}, name: {field.get('name', None)}, dataType: {field.get('dataType', None)}")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. All references use the `@id` field.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  - Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print(f"  - No records found.")

# Show sample from the first non-empty DataFrame
for rsid, df in dataframes.items():
    if not df.empty:
        print(f"\nSample data from {rsid}:")
        display(df.head())
        break

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize and group records in a chosen record set, always referencing fields by `@id`.

- Example: Filter records by a numeric field (e.g., log likelihood or coefficient), normalize it, and group by a categorical variable if present.

In [ ]:
# Select a record set and numeric field for analysis
# (Replace <rsid> and <fieldid> below with choices from the overview code output)
if dataframes:
    example_rsid = next(iter(dataframes))  # Just pick the first loaded record set
    df = dataframes[example_rsid]
    print(f"Using record set: {example_rsid}")
    print("Column names (possible field @ids):", list(df.columns))
    
    # Try to auto-choose a numeric field
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        print("No numeric field found for EDA.")
    else:
        print(f"Using numeric field: {numeric_field}")

        # Example: filter records with value > threshold (use median if not sure of scale)
        threshold = df[numeric_field].median()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold} (count: {len(filtered_df)}):")
        display(filtered_df.head())

        # Normalize selected numeric field
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Try grouping by a categorical/non-numeric field if available
        group_field = None
        for col in df.columns:
            if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col]):
                group_field = col
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field}:")
            display(grouped_df)
else:
    print("No record sets loaded for EDA.")

## 5. Visualization
Visualize numeric field distributions or relationships in the dataset, referencing fields by their `@id` where possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field histogram and group relations
if 'filtered_df' in locals() and numeric_field is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field} (after filtering)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If grouping field is present
    if group_field is not None:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- Using `mlcroissant`, we can programmatically explore and analyze Croissant-based datasets using stable, precise references via `@id` for record sets and fields.
- This dataset documents ordered logistic regression results for adoption predictors of indigenous and modern knowledge in rangeland management across sampled Northern Kenya counties.
- Further domain-specific analyses are possible by referencing fields and record sets by their Croissant `@id` in all code and documentation, ensuring robust and reproducible data workflows.